# V008 training adapters: unchanged V005 epoch engine, isolated per run

Definitions only; executing these cells reads no assignment data and launches no fits.
The public `fit_validate_v008`, `fit_fixed_v008`, `load_checkpoint_v008`, and
`predict_checkpoint_v008` accept a custom `model_factory(spec, n_features, prep)`.
The factory must return a scalar pressure model consuming V005 transformed inputs
(standardized features followed by the internal R/C code). Its fitted, training-only
preprocessor is available during both initial construction and checkpoint reconstruction.

Each invocation executes the immutable V005 notebook definitions in its own private
namespace. Only that namespace's preparation, loss, fingerprint and payload hooks are
adapted. The original optimizer, shuffling, complete epoch loops, frozen schedules,
best/last saves, scaler fitting, selection metric, and RNG restoration stay intact.
There is no global monkeypatch and no change to `sequence_components_v005.ipynb`.

`loss_kind='l1'` uses original scalar inspiratory L1. `loss_kind='curve_l1'` adds
exactly .10 times adjacent pressure-increment L1 where **both** positions are
inspiratory. Both terms are computed in linearly scaled target units, so their sum
is the native-unit objective divided by the one training-only target scale. Empty
adjacent masks contribute an exact differentiable zero. Selection and reported
primary MAE always remain native, point-weighted inspiratory MAE.

V008 checkpoints identify the arm, loss and factory, save the full preprocessor,
verify source/config/data/statistics hashes, and retain the V005 optimizer,
scheduler, Python/NumPy/Torch/CUDA and loader-generator states. Resume requires the
same numeric runtime and device identity. Trusted-local Torch serialization must
only be loaded from trusted artifacts. Prediction reconstructs without labels and
may use a different inference device. Supply model notebook code hashes through
`source_fingerprints` for every custom factory; notebook outputs are excluded from
automatic engine/trainer source hashes. Keep these source definitions immutable
within a stage.


In [ ]:
# Imports and immutable implementation identity; no file reads at definition time.
import copy
import hashlib
import json
import platform
import tempfile
import types
from pathlib import Path

import numpy as np
import torch
from torch import nn

V008_TRAINER_VERSION = 'v008-isolated-v005-curve-l1-1'
V008_CURVE_WEIGHT = .10


def _v8_notebook_path(name):
    for root in (Path.cwd(), *Path.cwd().parents):
        for candidate in (root/'research'/name, root/name):
            if candidate.is_file():
                return candidate.resolve()
    raise FileNotFoundError(f'Cannot locate research/{name} from the current working directory')


def notebook_code_sha256_v008(path):
    """Hash executable code, independent of JSON formatting, metadata and outputs."""
    book = json.loads(Path(path).read_text())
    source = '\n# --- notebook cell ---\n'.join(
        ''.join(cell.get('source', [])) for cell in book['cells']
        if cell.get('cell_type') == 'code')
    return hashlib.sha256(source.encode()).hexdigest()


def _v8_jsonable(value):
    if isinstance(value, dict):
        return {str(key): _v8_jsonable(item) for key, item in sorted(value.items(), key=lambda pair: str(pair[0]))}
    if isinstance(value, (list, tuple)):
        return [_v8_jsonable(item) for item in value]
    if isinstance(value, np.ndarray):
        return dict(array_shape=list(value.shape), dtype=value.dtype.str, values=value.tolist())
    if isinstance(value, np.generic):
        return value.item()
    return value


def _v8_statistics_sha256(prep):
    encoded = json.dumps(_v8_jsonable(prep), sort_keys=True, separators=(',', ':'), allow_nan=False)
    return hashlib.sha256(encoded.encode()).hexdigest()


def _v8_callable_identity(factory):
    """Stable callable-code identity; declared notebook hashes cover dependent classes."""
    if factory is None:
        return dict(module='v005', qualname='build_model', code_sha256=None)
    if not callable(factory):
        raise TypeError('model_factory must be callable or None')
    def stable(value):
        if isinstance(value, types.CodeType):
            return dict(code=value.co_code.hex(), names=list(value.co_names),
                varnames=list(value.co_varnames), constants=[stable(v) for v in value.co_consts])
        if value is None or isinstance(value, (str, bool, int, float)):
            return value
        if isinstance(value, (list, tuple)):
            return [stable(v) for v in value]
        if isinstance(value, dict):
            return {str(k): stable(v) for k, v in sorted(value.items(), key=lambda pair: str(pair[0]))}
        return dict(type=type(value).__module__+'.'+type(value).__qualname__)
    code = getattr(factory, '__code__', getattr(getattr(factory, '__call__', None), '__code__', None))
    closure = getattr(factory, '__closure__', None)
    identity = dict(code=stable(code), defaults=stable(getattr(factory, '__defaults__', None)),
        kwdefaults=stable(getattr(factory, '__kwdefaults__', None)),
        closure=None if closure is None else [stable(cell.cell_contents) for cell in closure])
    encoded = json.dumps(identity, sort_keys=True, separators=(',', ':'), allow_nan=False)
    return dict(module=getattr(factory, '__module__', type(factory).__module__),
        qualname=getattr(factory, '__qualname__', type(factory).__qualname__),
        code_sha256=hashlib.sha256(encoded.encode()).hexdigest())


def _v8_metadata(arm, loss_kind, model_factory, source_fingerprints):
    if not isinstance(arm, str) or not arm.strip():
        raise ValueError('A nonempty versioned arm identity is required')
    if loss_kind not in ('l1', 'curve_l1'):
        raise ValueError('loss_kind must be l1 or curve_l1')
    sources = {} if source_fingerprints is None else dict(source_fingerprints)
    if model_factory is not None and not sources:
        raise ValueError('Custom factories require declared model source_fingerprints')
    for key, digest in sources.items():
        if not isinstance(key, str) or not isinstance(digest, str) or len(digest) != 64 or any(c not in '0123456789abcdef' for c in digest):
            raise ValueError('Source fingerprints must map names to lowercase SHA256 digests')
    return dict(trainer_version=V008_TRAINER_VERSION, arm=arm, loss_kind=loss_kind,
        curve_aux_weight=V008_CURVE_WEIGHT if loss_kind == 'curve_l1' else 0.,
        engine_code_sha256=notebook_code_sha256_v008(_v8_notebook_path('sequence_components_v005.ipynb')),
        trainer_code_sha256=notebook_code_sha256_v008(_v8_notebook_path('training_components_v008.ipynb')),
        source_fingerprints=dict(sorted(sources.items())),
        factory_identity=_v8_callable_identity(model_factory))


def _v8_runtime(device):
    selected = torch.device(device)
    result = dict(python=platform.python_version(), numpy=np.__version__, torch=torch.__version__,
        torch_cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(), device_type=selected.type,
        deterministic=True, thread_count=4)
    if selected.type == 'cuda':
        result.update(cuda_device_count=torch.cuda.device_count(),
            cuda_device_index=torch.cuda.current_device() if selected.index is None else selected.index,
            cuda_device_name=torch.cuda.get_device_name(selected))
    return result


def curve_l1_loss_v008(prediction, target, inspiration_mask):
    """Return (main+.10*increment, main, increment), all in target-scaled units.

    Baseline profiles and target centering cancel in the prediction-minus-target
    increment; the shared positive target scale multiplies both native terms.
    """
    if prediction.ndim != 2 or target.shape != prediction.shape or inspiration_mask.shape != prediction.shape:
        raise ValueError('Expected matching [breaths,steps] prediction, target and mask')
    mask = inspiration_mask.bool()
    difference = prediction-target
    if not bool(mask.any()):
        raise ValueError('Main L1 requires at least one inspiratory point')
    main = difference.masked_select(mask).abs().mean()
    adjacent_mask = mask[:, 1:] & mask[:, :-1]
    if bool(adjacent_mask.any()):
        increment_error = ((prediction[:, 1:]-prediction[:, :-1])
            -(target[:, 1:]-target[:, :-1]))
        adjacent = increment_error.masked_select(adjacent_mask).abs().mean()
    else:
        adjacent = prediction[:, :0].sum()
    return main+V008_CURVE_WEIGHT*adjacent, main, adjacent


In [ ]:
# Per-invocation adapter; every V005 function keeps this private namespace.
def _v8_engine(arm, loss_kind, model_factory, source_fingerprints, device):
    metadata = _v8_metadata(arm, loss_kind, model_factory, source_fingerprints)
    engine_path = _v8_notebook_path('sequence_components_v005.ipynb')
    book = json.loads(engine_path.read_text())
    namespace = {'__name__': '_v008_private_sequence_engine'}
    for index, cell in enumerate(book['cells']):
        if cell.get('cell_type') == 'code':
            exec(compile(''.join(cell.get('source', [])), f'{engine_path}:cell{index}', 'exec'), namespace)
    namespace['_sequence_device'](device)
    original_fingerprint = namespace['_fingerprint']
    original_payload = namespace['_checkpoint_payload']
    original_configuration = namespace['_train_configuration']
    original_build_model = namespace['build_model']
    runtime = _v8_runtime(device)
    latest_statistics = {}

    def configuration(train_cfg):
        cfg = original_configuration(train_cfg)
        if cfg['loss'] != 'l1':
            raise ValueError('V008 scalar arms require train_cfg.loss=l1; select the auxiliary term with loss_kind')
        return cfg

    def fingerprint(raw, y, out, spec, cfg, seed, feature_names, mode, extra=None):
        result = original_fingerprint(raw, y, out, spec, cfg, seed, feature_names, mode, extra)
        result.update(v008=copy.deepcopy(metadata), numeric_runtime=copy.deepcopy(runtime))
        return result

    def payload(model, prep, spec, cfg, seed, run_fingerprint, optimizer, scheduler,
                generator, epoch, history, completed=False, termination=None):
        if history and int(history[-1]['epoch']) == int(epoch):
            history[-1].update(copy.deepcopy(latest_statistics))
        result = original_payload(model, prep, spec, cfg, seed, run_fingerprint,
            optimizer, scheduler, generator, epoch, history, completed, termination)
        result.update(format_version=8, base_engine_format_version=5,
            v008=copy.deepcopy(metadata), preprocessing_sha256=_v8_statistics_sha256(prep))
        return result

    def prepare(raw, y, out, spec, cfg, seed, feature_names, checkpoint_path, resume,
                run_fingerprint, selected_device):
        checkpoint_path, last = Path(checkpoint_path), namespace['_last_path'](checkpoint_path)
        if not resume and (checkpoint_path.exists() or last.exists()):
            raise FileExistsError('Use a fresh path or explicitly resume an existing stage')
        namespace['seed_sequence'](seed)
        if resume:
            if not last.is_file():
                raise FileNotFoundError(f'Missing last-state checkpoint: {last}')
            saved = torch.load(last, map_location='cpu', weights_only=False)
            if saved.get('format_version') != 8 or saved.get('v008') != metadata:
                raise ValueError('V008 resume rejected: arm, loss, factory, or source implementation changed')
            prep = namespace['_validated_preprocessor'](saved['preprocess'])
            prep_hash = _v8_statistics_sha256(prep)
            if saved.get('preprocessing_sha256') != prep_hash:
                raise ValueError('V008 resume rejected: fitted preprocessing checksum differs')
            run_fingerprint['preprocessing_sha256'] = prep_hash
            if saved.get('fingerprint') != run_fingerprint:
                raise ValueError('V008 resume rejected: stage data, recipe, seed, budget, statistics, or runtime changed')
        else:
            saved = None
            prep = namespace['fit_preprocessor'](raw, y, out, spec, feature_names,
                scaler_phase=cfg['scaler_phase'], residual_baseline=cfg['residual_baseline'])
            run_fingerprint['preprocessing_sha256'] = _v8_statistics_sha256(prep)
        model = (original_build_model(spec, raw.shape[-1]) if model_factory is None
            else model_factory(copy.deepcopy(spec), raw.shape[-1], copy.deepcopy(prep)))
        if not isinstance(model, nn.Module):
            raise TypeError('model_factory must return a torch.nn.Module')
        model = model.to(selected_device)
        optimizer = namespace['_optimizer'](model, cfg)
        loader, generator, transformed = namespace['_loader'](raw, y, out, prep, cfg, seed, selected_device)
        return model, optimizer, loader, generator, transformed, prep, saved

    def training_epoch(model, loader, optimizer, cfg, prep, selected_device):
        model.train()
        total_error, total_count, optimizer_steps = 0., 0, 0
        adjacent_error, adjacent_count, objective_sum = 0., 0, 0.
        for xb, yb, mask in loader:
            xb, yb, mask = (value.to(selected_device, non_blocking=selected_device.type == 'cuda')
                for value in (xb, yb, mask))
            if not bool(mask.any()):
                continue
            optimizer.zero_grad(set_to_none=True)
            prediction = model(xb)
            if prediction.shape != yb.shape:
                raise ValueError('Scalar V008 model must return [breaths,80]')
            difference = (prediction-yb).masked_select(mask)
            if loss_kind == 'curve_l1':
                loss, main, adjacent = curve_l1_loss_v008(prediction, yb, mask)
                pair_count = int((mask[:, 1:] & mask[:, :-1]).sum())
                adjacent_error += float(adjacent.detach())*pair_count*prep['target_scale']
                adjacent_count += pair_count
            else:
                loss = difference.abs().mean()
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite training loss')
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), float(cfg['gradient_clip']))
            optimizer.step()
            optimizer_steps += 1
            total_error += float(difference.detach().abs().sum())*prep['target_scale']
            total_count += difference.numel()
            objective_sum += float(loss.detach())*difference.numel()
        if not total_count:
            raise ValueError('Training epoch contained no scored points')
        latest_statistics.clear()
        latest_statistics.update(training_loss_kind=loss_kind,
            training_online_objective_scaled=objective_sum/total_count,
            training_adjacent_inspiratory_pairs=adjacent_count,
            training_online_increment_mae=(adjacent_error/adjacent_count if adjacent_count else 0.))
        return float(total_error/total_count), total_count, optimizer_steps

    namespace.update(_train_configuration=configuration, _fingerprint=fingerprint,
        _checkpoint_payload=payload, _prepare_run=prepare, _training_epoch=training_epoch)
    return namespace, metadata


def _v8_report(report, metadata):
    report = copy.deepcopy(report)
    report.update(format_version=8, v008=copy.deepcopy(metadata), arm=metadata['arm'],
        loss_kind=metadata['loss_kind'], curve_aux_weight=metadata['curve_aux_weight'])
    return report


def fit_validate_v008(x_train, y_train, u_out_train, x_val, y_val, u_out_val,
                      model_spec, train_cfg, seed, checkpoint_path, epoch_log_path=None,
                      feature_names=None, train_diagnostic_breaths=2048, device='cuda',
                      resume=False, on_epoch=None, *, model_factory=None, loss_kind='l1',
                      arm='v008_scalar', source_fingerprints=None):
    """V005 A selection unchanged; the custom arm/loss/source identity is immutable."""
    namespace, metadata = _v8_engine(arm, loss_kind, model_factory, source_fingerprints, device)
    report = namespace['fit_validate'](x_train, y_train, u_out_train, x_val, y_val, u_out_val,
        model_spec, train_cfg, seed, checkpoint_path, epoch_log_path=epoch_log_path,
        feature_names=feature_names, train_diagnostic_breaths=train_diagnostic_breaths,
        device=device, resume=resume, on_epoch=on_epoch)
    return _v8_report(report, metadata)


def fit_fixed_v008(x_train, y_train, u_out_train, model_spec, train_cfg, seed, lr_schedule,
                   checkpoint_path, epoch_log_path=None, feature_names=None, device='cuda',
                   resume=False, on_epoch=None, *, model_factory=None, loss_kind='l1',
                   arm='v008_scalar', source_fingerprints=None):
    """V005 frozen-budget fit unchanged; accepts no validation arrays or labels."""
    namespace, metadata = _v8_engine(arm, loss_kind, model_factory, source_fingerprints, device)
    report = namespace['fit_fixed'](x_train, y_train, u_out_train, model_spec, train_cfg,
        seed, lr_schedule, checkpoint_path, epoch_log_path=epoch_log_path,
        feature_names=feature_names, device=device, resume=resume, on_epoch=on_epoch)
    return _v8_report(report, metadata)


def load_checkpoint_v008(checkpoint_path, device='cpu', *, model_factory=None,
                         expected_arm=None, expected_loss_kind=None, source_fingerprints=None):
    """Reconstruct a trusted V008 checkpoint using only saved spec/statistics and code."""
    saved = torch.load(Path(checkpoint_path), map_location='cpu', weights_only=False)
    if saved.get('format_version') != 8 or not isinstance(saved.get('v008'), dict):
        raise ValueError('Expected a V008 scalar checkpoint')
    metadata = saved['v008']
    if expected_arm is not None and expected_arm != metadata['arm']:
        raise ValueError('Checkpoint arm differs from expected_arm')
    if expected_loss_kind is not None and expected_loss_kind != metadata['loss_kind']:
        raise ValueError('Checkpoint loss differs from expected_loss_kind')
    namespace, current = _v8_engine(metadata['arm'], metadata['loss_kind'], model_factory,
        source_fingerprints, device)
    if metadata != current or saved.get('fingerprint', {}).get('v008') != metadata:
        raise ValueError('Checkpoint source implementation or factory identity differs')
    prep = namespace['_validated_preprocessor'](saved['preprocess'])
    digest = _v8_statistics_sha256(prep)
    if saved.get('preprocessing_sha256') != digest or saved['fingerprint'].get('preprocessing_sha256') != digest:
        raise ValueError('Checkpoint fitted preprocessing checksum differs')
    model = (namespace['build_model'](saved['model_spec'], prep['input_shape'][1])
        if model_factory is None else model_factory(copy.deepcopy(saved['model_spec']),
            prep['input_shape'][1], copy.deepcopy(prep)))
    if not isinstance(model, nn.Module):
        raise TypeError('model_factory must return a torch.nn.Module')
    model = model.to(namespace['_sequence_device'](device))
    model.load_state_dict(saved['state_dict'], strict=True)
    model.eval()
    return model, prep, saved


def predict_checkpoint_v008(checkpoint_path, x, device='cpu', batch_size=512, *,
                            model_factory=None, expected_arm=None, expected_loss_kind=None,
                            source_fingerprints=None):
    """Label-free, native-unit prediction with independent state for every breath."""
    if int(batch_size) < 1:
        raise ValueError('batch_size must be positive')
    model, prep, saved = load_checkpoint_v008(checkpoint_path, device=device,
        model_factory=model_factory, expected_arm=expected_arm,
        expected_loss_kind=expected_loss_kind, source_fingerprints=source_fingerprints)
    # These prediction-only helpers come from a fresh namespace with no shared hooks.
    namespace, _ = _v8_engine(saved['v008']['arm'], saved['v008']['loss_kind'],
        model_factory, source_fingerprints, device)
    return namespace['_predict_native'](model, namespace['transform_inputs'](x, prep),
        prep, namespace['_sequence_device'](device), int(batch_size))


In [ ]:
# Explicit synthetic validation only; never called by notebook definition/import.
def v8_training_selfcheck(model_factory=None, arm='v008_curve_selfcheck',
                          source_fingerprints=None, device='cpu', loss_kind='curve_l1',
                          model_spec=None, n_features=47, feature_names=None):
    """Exercise native prediction, loss boundaries and exact epoch-boundary resume.

    Call explicitly on CPU and the actual training GPU. CUDA is never emulated and
    deterministic execution is never relaxed to make a check pass. Tiny synthetic
    whole breaths and temporary checkpoints are used; no assignment files are read.
    """
    checks = {}
    def check(name, condition):
        if not bool(condition):
            raise AssertionError(name)
        checks[name] = True
    def equal(left, right):
        if torch.is_tensor(left):
            return torch.is_tensor(right) and torch.equal(left, right)
        if isinstance(left, np.ndarray):
            return isinstance(right, np.ndarray) and np.array_equal(left, right)
        if isinstance(left, dict):
            return isinstance(right, dict) and left.keys() == right.keys() and all(equal(left[k], right[k]) for k in left)
        if isinstance(left, (list, tuple)):
            return type(left) is type(right) and len(left) == len(right) and all(equal(a, b) for a, b in zip(left, right))
        return left == right
    def rejects(name, callback):
        try:
            callback()
        except (ValueError, FileExistsError):
            checks[name] = True
        else:
            raise AssertionError(name)

    selected = torch.device(device)
    if selected.type == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('Actual CUDA required for the GPU selfcheck')
    n_features = int(n_features)
    if n_features < 7:
        raise ValueError('Synthetic arm checks need at least the seven base input channels')
    names = (['R', 'C', 'time_step', 'u_in', 'u_out', 'step', 'dt']
        +[f'toy_feature_{i}' for i in range(7, n_features)]) if feature_names is None else list(feature_names)
    if len(names) != n_features or names[:2] != ['R', 'C']:
        raise ValueError('Supply one raw feature name per channel, beginning with R,C')
    rng = np.random.default_rng(20261108)
    x = rng.normal(0., .2, (8, 80, n_features)).astype(np.float32)
    x[..., 0] = np.array([5, 20, 50, 5, 20, 50, 5, 20], np.float32)[:, None]
    x[..., 1] = np.array([10, 20, 50, 20, 50, 10, 50, 10], np.float32)[:, None]
    steps = np.arange(80, dtype=np.float32)[None, :]
    dt = np.broadcast_to(.025+.0001*(steps % 5), (8, 80)).copy()
    dt[:, 0] = 0.
    times = np.cumsum(dt, axis=1)
    control = 8.+2.*np.sin(steps/7.)+np.arange(8, dtype=np.float32)[:, None]*.3
    out = np.broadcast_to(steps >= 31, (8, 80)).astype(np.int8).copy()
    for name, values in [('time_step', times), ('u_in', control), ('u_out', out),
                         ('step', np.broadcast_to(steps, (8, 80))), ('dt', dt)]:
        if name in names:
            x[..., names.index(name)] = values
    y = (3.+.13*control+.07*steps+.1*np.arange(8, dtype=np.float32)[:, None]
        +.03*rng.normal(size=(8, 80))).astype(np.float32)
    spec = dict(family='lstm', hidden=8, layers=2, dropout=.15,
        bidirectional=True, head_width=8) if model_spec is None else copy.deepcopy(model_spec)
    cfg = dict(lr=.0015, batch_size=2, max_epochs=3, patience=0, min_delta=0.,
        scheduler='replay', frozen_lr_schedule=[.0015, .001, .0005],
        prediction_batch_size=2, optimizer='Adam', weight_decay=0., loss='l1')
    common = dict(model_spec=spec, train_cfg=cfg, seed=20261102, feature_names=names,
        device=str(selected), model_factory=model_factory, arm=arm, loss_kind=loss_kind,
        source_fingerprints=source_fingerprints)
    prediction_options = dict(device=str(selected), batch_size=2, model_factory=model_factory,
        expected_arm=arm, expected_loss_kind=loss_kind, source_fingerprints=source_fingerprints)

    # Numeric loss checks include nonconstant baseline: it cancels from error increments.
    p = torch.tensor([[.2, .4, -.3, .1]], device=selected, requires_grad=True)
    t = torch.tensor([[.1, .2, .7, -.4]], device=selected)
    mask = torch.tensor([[True, True, False, True]], device=selected)
    loss, main, adjacent = curve_l1_loss_v008(p, t, mask)
    check('curve_uses_only_both_inspiratory_neighbors', torch.allclose(adjacent, p.new_tensor(.1)))
    scale, center = 7., 4.
    baseline = p.new_tensor([[.3, -.7, .8, .2]])
    native, native_main, native_adjacent = curve_l1_loss_v008(p*scale+center+baseline,
        t*scale+center+baseline, mask)
    check('scaled_curve_objective_native_linear_equivalent', torch.allclose(loss*scale, native, rtol=1e-5, atol=1e-6))
    isolated = torch.tensor([[True, False, True, False]], device=selected)
    empty_loss, empty_main, empty_aux = curve_l1_loss_v008(p, t, isolated)
    check('empty_adjacent_mask_exact_zero_finite', float(empty_aux.detach()) == 0. and bool(torch.isfinite(empty_loss)))
    empty_loss.backward()
    check('empty_adjacent_backward_finite', bool(torch.isfinite(p.grad).all()))
    check('expiration_pressure_excluded_from_main_and_aux',
        torch.equal(curve_l1_loss_v008(p, t.masked_fill(~mask, 9999.), mask)[0], loss))

    class Interrupted(RuntimeError):
        pass
    def stop_after_first(record):
        if record['epoch'] == 1:
            raise Interrupted('Synthetic interruption after authoritative epoch checkpoint')

    summaries = {}
    with tempfile.TemporaryDirectory(prefix='v008-training-check-') as directory:
        for mode in ('validate', 'fixed'):
            full, resumed = Path(directory)/f'{mode}-full.pt', Path(directory)/f'{mode}-resumed.pt'
            if mode == 'validate':
                def run(path, resume=False, on_epoch=None, **overrides):
                    kwargs = dict(common, checkpoint_path=path, resume=resume, on_epoch=on_epoch,
                        train_diagnostic_breaths=4)
                    kwargs.update(overrides)
                    return fit_validate_v008(x[:6], y[:6], out[:6], x[6:], y[6:], out[6:], **kwargs)
            else:
                def run(path, resume=False, on_epoch=None, **overrides):
                    kwargs = dict(common, checkpoint_path=path, resume=resume, on_epoch=on_epoch,
                        lr_schedule=cfg['frozen_lr_schedule'])
                    kwargs.update(overrides)
                    return fit_fixed_v008(x[:6], y[:6], out[:6], **kwargs)
            full_report = run(full)
            try:
                run(resumed, on_epoch=stop_after_first)
            except Interrupted:
                checks[f'{mode}_interruption_after_saved_epoch'] = True
            else:
                raise AssertionError('Synthetic interruption did not trigger')
            resumed_report = run(resumed, resume=True)
            a = torch.load(full.with_name(full.stem+'.last.pt'), map_location='cpu', weights_only=False)
            b = torch.load(resumed.with_name(resumed.stem+'.last.pt'), map_location='cpu', weights_only=False)
            for key in ('state_dict', 'optimizer_state', 'scheduler_state', 'rng_state',
                        'preprocess', 'fingerprint', 'model_spec', 'config', 'epoch', 'optimizer_steps'):
                check(f'{mode}_resume_exact_{key}', equal(a[key], b[key]))
            metrics = ('lr_used', 'training_online_inspiratory_mae', 'optimizer_steps',
                'training_online_objective_scaled', 'training_online_increment_mae')
            if mode == 'validate':
                metrics += ('validation_inspiratory_mae', 'training_eval_inspiratory_mae', 'best_epoch')
            check(f'{mode}_resume_exact_epoch_metrics', all(
                all(equal(aa[k], bb[k]) for k in metrics)
                for aa, bb in zip(full_report['epochs'], resumed_report['epochs'])))
            pa = predict_checkpoint_v008(full, x[6:], **prediction_options)
            pb = predict_checkpoint_v008(resumed, x[6:], **prediction_options)
            check(f'{mode}_resume_exact_native_prediction', np.array_equal(pa, pb))
            check(f'{mode}_native_prediction_shape_finite', pa.shape == (2, 80) and np.isfinite(pa).all())
            single = predict_checkpoint_v008(full, x[6:7], **prediction_options)
            check(f'{mode}_independent_breath_inference', np.allclose(single, pa[:1], rtol=1e-5, atol=1e-6))
            check(f'{mode}_reports_complete_budget', full_report['completed'] and resumed_report['completed']
                and full_report['executed_epochs'] == resumed_report['executed_epochs'] == 3)
            rejects(f'{mode}_reject_overwrite_without_resume', lambda: run(full))
            rejects(f'{mode}_reject_changed_arm', lambda: run(resumed, resume=True, arm=arm+'_changed'))
            changed_loss = 'l1' if loss_kind == 'curve_l1' else 'curve_l1'
            rejects(f'{mode}_reject_changed_loss', lambda: run(resumed, resume=True, loss_kind=changed_loss))
            changed_sources = dict(source_fingerprints or {})
            changed_sources['deliberately_changed_source'] = '0'*64
            rejects(f'{mode}_reject_changed_source', lambda: run(resumed, resume=True, source_fingerprints=changed_sources))
            changed_cfg = dict(cfg, gradient_clip=.5)
            rejects(f'{mode}_reject_changed_configuration', lambda: run(resumed, resume=True, train_cfg=changed_cfg))
            if mode == 'validate':
                check('selection_metric_is_native_inspiratory_mae', np.isclose(full_report['validation_mae'],
                    np.abs(pa.astype(np.float64)-y[6:].astype(np.float64))[out[6:] == 0].mean(), rtol=1e-6, atol=1e-6))
            else:
                check('fixed_budget_has_no_validation_labels', not full_report['validation_used'])
            summaries[mode] = dict(epochs=3, optimizer_steps=a['optimizer_steps'], parameter_count=a['parameter_count'])
    return dict(scope='Synthetic whole-breath V008 training/resume checks only; no assignment data',
        passed=all(checks.values()), check_count=len(checks), checks=checks,
        arm=arm, loss_kind=loss_kind, device=str(selected), runtime=_v8_runtime(selected),
        model_spec=spec, n_features=n_features, fit_modes=summaries,
        gpu_checked=selected.type == 'cuda')


Usage follows the V005 positional arguments, with the explicit V008 keywords:

```python
report = fit_validate_v008(
    x_train, y_train, out_train, x_val, y_val, out_val,
    compact_spec, train_cfg, seed, checkpoint_path,
    feature_names=names, model_factory=arm_factory,
    arm='v008_pressure_curve', loss_kind='curve_l1',
    source_fingerprints=model_code_hashes, device='cuda')
```

The pressure-curve arm uses the original compact scalar model factory. The learned
memory and residual attention arms use their separate custom factories and
`loss_kind='l1'`. No combined arm is introduced. Frozen confirmation/final stages
use `fit_fixed_v008` with the selected A schedule and the same factory identity.

Run `v8_training_selfcheck(...)` explicitly for each arm on CPU and on the actual
training GPU before any assignment-data fits. `gpu_checked=False` is an honest CPU
result, never evidence that CUDA deterministic operations or GPU resume passed.
